### ЗАДАЧА: Распределение доставок между курьерами

Логистическая команда получает пакет заявок на доставки, которые нужно распределить между курьерами на электровелосипедах.
Нужно собрать систему, которая:
- принимает корректные доставки,
- отбрасывает неправильные или небезопасные заявки,
- уменьшает доступный заряд после успешно назначенного маршрута,
- ведёт отдельный журнал ошибок,
- помогает понять, какой курьер был загружен дольше всех и какому клиенту доставили самый большой вес.

In [7]:
from dataclasses import dataclass
from typing import Optional


couriers = {
    'CR-1': {'zone': 'north', 'charge_min': 40, 'max_weight': 3.0},
    'CR-2': {'zone': 'south', 'charge_min': 30, 'max_weight': 2.0},
    'CR-3': {'zone': 'north', 'charge_min': 55, 'max_weight': 5.0},
}

# rows: delivery_id|courier_id|client|weight_kg|route_min
rows = [
    'DL-100|CR-1|Clinic|1.5|12',
    'DL-101|CR-2|Cafe|2.5|10',
    'DL-102|CR-9|Lab|1.0|8',
    'DL-103|CR-1|Shop|0|6',
    'DL-104|CR-3|Village|3.5|60',
    'DL-100|CR-3|Clinic|1.0|10',
    'DL-105|CR-3|School|2.0|20',
    'DL-106|CR-2|Pharmacy|1.0|15',
]


class DeliveryError(Exception):
    pass


class RowFormatError(DeliveryError):
    pass


class CourierNotFoundError(DeliveryError):
    pass


class WeightError(DeliveryError):
    pass


class RouteTimeError(DeliveryError):
    pass


class WeightLimitError(DeliveryError):
    pass


class ChargeReserveError(DeliveryError):
    pass


class DuplicateDeliveryError(DeliveryError):
    pass


@dataclass(order=True)
class Delivery:
    route_min: int
    delivery_id: str
    courier_id: str
    client: str
    weight_kg: float


class Courier:
    def __init__(self, courier_id, zone, charge_min, max_weight):
        # TODO: сохранить courier_id, zone, charge_min, max_weight
        # TODO: создать список deliveries
        self.courier_id = courier_id
        self.zone = zone
        self.charge_min = charge_min
        self.max_weight = max_weight
        self.deliveries = []

    def charge_left(self):
        # TODO: вернуть текущий остаток заряда в минутах
        return self.charge_min

    def total_route_time(self):
        # TODO: вернуть сумму route_min по self.deliveries
        m = 0
        for dv in self.deliveries:
            m += dv.route_min
        return m


    def total_weight(self):
        # TODO: вернуть сумму weight_kg по self.deliveries
        m = 0
        for dv in self.deliveries:
            m += dv.weight_kg
        return m
    def assign(self, delivery):
        # TODO: если delivery.weight_kg > self.max_weight -> raise WeightLimitError(...)
        # TODO: посчитать charge_after = charge_left() - delivery.route_min
        # TODO: если charge_after < 5 -> raise ChargeReserveError(...)
        # TODO: добавить delivery в self.deliveries
        # TODO: отсортировать self.deliveries
        if delivery.weight_kg > self.max_weight:
            raise WeightLimitError('вес заказа больше максимального!')
        charge_after = self.charge_left() - delivery.route_min
        if charge_after < 5:
            raise ChargeReserveError('charge_after < 5')
        self.deliveries.append(delivery)
        self.deliveries.sort()



class CourierDispatchService:
    def __init__(self, couriers):
        # TODO: создать couriers вида courier_id -> Courier(...)
        # TODO: создать списки accepted и errors
        # TODO: создать множество processed_ids
        self.couriers = {}
        for key, val in couriers.items():
            self.couriers.setdefault(key, Courier(courier_id = key, zone=val['zone'], charge_min=val['charge_min'], max_weight=val['max_weight']))
        self.accepted = []
        self.errors = []
        self.processed_ids = set()

    def parse_delivery(self, row):
        # TODO: split по '|'
        # TODO: ожидать 5 частей: delivery_id, courier_id, client, weight_raw, route_raw
        # TODO: если частей не 5 -> raise RowFormatError(...)
        # TODO: проверить, что courier_id существует
        # TODO: weight_raw преобразовать в float
        # TODO: route_raw преобразовать в int
        # TODO: ошибки преобразования поднимать через WeightError / RouteTimeError с raise ... from exc
        # TODO: если weight_kg <= 0 -> raise WeightError(...)
        # TODO: если route_min <= 0 -> raise RouteTimeError(...)
        # TODO: вернуть Delivery(...)
        arr = row.split('|')
        if len(arr) != 5:
            raise RowFormatError('неверный формат!')
        delivery_id, courier_id, client, weight_raw, route_raw = arr[0], arr[1], arr[2], arr[3], arr[4]
        try:
            weight_raw = float(weight_raw)
        except ValueError as e:
            raise WeightError('неккоректный вес!') from e
        try:
            route_raw = int(route_raw)
        except ValueError as e:
            raise RouteTimeError('неккоректное route_raw!') from e
        if weight_raw <= 0:
            raise WeightError('weight_kg должен быть > 0!')
        if route_raw <= 0:
            raise RouteTimeError('route_min должно быть > 0!')
        if courier_id not in self.couriers.keys():
            raise CourierNotFoundError('такого courier_id нет!')
        return Delivery(delivery_id=delivery_id, courier_id=courier_id, client=client, weight_kg=weight_raw, route_min=route_raw)

    def submit(self, row):
        # TODO: внутри try вызвать parse_delivery(row)
        # TODO: если delivery.delivery_id уже в processed_ids -> raise DuplicateDeliveryError(...)
        # TODO: передать delivery в couriers[delivery.courier_id].assign(delivery)
        # TODO: после успеха обновить processed_ids и accepted
        # TODO: DeliveryError сохранить в errors как (row, error_type, message)
        try:
            dv = self.parse_delivery(row)
            if dv.delivery_id in self.processed_ids:
                raise DuplicateDeliveryError('delivery_id уже обработано!')
            self.couriers[dv.courier_id].assign(dv)
            self.processed_ids.add(dv.delivery_id)
            self.accepted.append(dv)
        except DeliveryError as e:
            self.errors.append((row, type(e).__name__, e))


    def load(self, rows):
        # TODO: вызвать submit(row) для каждой строки
        for el in rows:
            self.submit(el)

    def client_weights(self):
        # TODO: собрать dict вида client -> total_weight_kg
        res = {}
        for el in self.accepted:
            res.setdefault(el.client, 0)
            res[el.client] += el.weight_kg
        return res
    def top_client(self):
        # TODO: использовать client_weights()
        # TODO: вернуть tuple(client, weight_kg) с максимумом
        w = self.client_weights()
        res = None
        m = 0
        for key, val in w.items():
            if val > m:
                m = val
                res = (key, val)
        return res


    def busiest_courier(self):
        # TODO: найти курьера с максимумом total_route_time()
        # TODO: вернуть tuple(courier_id, total_route_time)
        m = 0
        courier = None
        for key, val in self.couriers.items():
            if val.total_route_time() > m:
                m = val.total_route_time()
                courier = key
        return (courier, m)


    def low_charge_couriers(self, threshold=15):
        # TODO: вернуть список tuple(courier_id, charge_left)
        # TODO: включать только курьеров, у которых charge_left() <= threshold
        res = []
        for key, val in self.couriers.items():
            if val.charge_left() <= threshold:
                res.append((key, val))
        return res



    def find_delivery(self, delivery_id):
        # TODO: пройтись по всем курьерам и их доставкам
        # TODO: если delivery.delivery_id совпал -> вернуть объект Delivery
        # TODO: если доставка не найдена -> вернуть None
        for key, val in self.couriers.items():
            for el in val.deliveries:
                if el.delivery_id == delivery_id:
                    return el
        return None


service = CourierDispatchService(couriers)

# TODO: загрузить rows через service.load(rows)
# TODO: вывести принятые доставки
# TODO: вывести ошибки
# TODO: вывести по каждому курьеру deliveries, total_route_time и charge_left
# TODO: вывести top_client()
# TODO: вывести busiest_courier()
# TODO: вывести low_charge_couriers()
# TODO: вывести find_delivery('DL-105')
        
service.load(rows)
print(service.accepted)
print(service.errors)
for key, val in service.couriers.items():
    print(f'Courier {key}: deliveries - {val.deliveries}; total_route_time - {val.total_route_time()}; charge_left - {val.charge_left()}')
print(service.top_client())
print(service.busiest_courier())
print(service.low_charge_couriers())
print(service.find_delivery('DL-105'))

[Delivery(route_min=12, delivery_id='DL-100', courier_id='CR-1', client='Clinic', weight_kg=1.5), Delivery(route_min=20, delivery_id='DL-105', courier_id='CR-3', client='School', weight_kg=2.0), Delivery(route_min=15, delivery_id='DL-106', courier_id='CR-2', client='Pharmacy', weight_kg=1.0)]
[('DL-101|CR-2|Cafe|2.5|10', 'WeightLimitError', WeightLimitError('вес заказа больше максимального!')), ('DL-102|CR-9|Lab|1.0|8', 'CourierNotFoundError', CourierNotFoundError('такого courier_id нет!')), ('DL-103|CR-1|Shop|0|6', 'WeightError', WeightError('weight_kg должен быть > 0!')), ('DL-104|CR-3|Village|3.5|60', 'ChargeReserveError', ChargeReserveError('charge_after < 5')), ('DL-100|CR-3|Clinic|1.0|10', 'DuplicateDeliveryError', DuplicateDeliveryError('delivery_id уже обработано!'))]
Courier CR-1: deliveries - [Delivery(route_min=12, delivery_id='DL-100', courier_id='CR-1', client='Clinic', weight_kg=1.5)]; total_route_time - 12; charge_left - 40
Courier CR-2: deliveries - [Delivery(route_min=